# 📊 Project 01 — Customer Retention & Revenue Analytics
## Step 1: Data Ingestion & Cleaning

**Goal:** Load all raw CSV files, check their quality, clean any problems, and save clean versions.

> ⚠️ **Golden Rule:** NEVER edit the original files. Always save cleaned data to a NEW folder.

---
### 📁 Folder Setup
Before running, make sure your CSV files are inside a folder called `data/raw/`
```
project/
├── data/
│   ├── raw/        ← put all your CSV files here
│   └── cleaned/    ← cleaned files will be saved here automatically
└── step01_data_cleaning.ipynb
```

## 📦 Cell 1: Import Libraries
We need two libraries:
- **pandas** → for reading and working with tables (like Excel in Python)
- **os** → for creating folders on your computer

In [1]:
import pandas as pd
import os

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


## 📂 Cell 2: Set Folder Paths & Create Output Folder

In [2]:
# Where your raw CSV files are stored
DATA_DIR = "./data/raw"

# Where cleaned files will be saved
OUTPUT_DIR = "./data/cleaned"

# Create the output folder if it doesn't exist yet
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📂 Raw data folder  : {DATA_DIR}")
print(f"📂 Output folder    : {OUTPUT_DIR}")
print("✅ Folders are ready!")

📂 Raw data folder  : ./data/raw
📂 Output folder    : ./data/cleaned
✅ Folders are ready!


## 📥 Cell 3: Load All Raw CSV Files
We load 7 files — 3 fact tables (things that happen) and 4 dimension tables (descriptions).

In [3]:
# --- DIMENSION TABLES (describe WHO and WHAT) ---
users   = pd.read_csv(f"{DATA_DIR}/dim_users.csv")
plans   = pd.read_csv(f"{DATA_DIR}/dim_plans.csv")
geo     = pd.read_csv(f"{DATA_DIR}/dim_geography.csv")
channel = pd.read_csv(f"{DATA_DIR}/dim_acquisition_channel.csv")

# --- FACT TABLES (record things that happened) ---
subs    = pd.read_csv(f"{DATA_DIR}/fact_subscriptions.csv")
pay     = pd.read_csv(f"{DATA_DIR}/fact_payments.csv")
act     = pd.read_csv(f"{DATA_DIR}/fact_activity_logs.csv")

# Print how many rows each file has
print("📊 Files Loaded Successfully!")
print("-" * 45)
print(f"  dim_users          : {len(users):>10,} rows")
print(f"  dim_plans          : {len(plans):>10,} rows")
print(f"  dim_geography      : {len(geo):>10,} rows")
print(f"  dim_channels       : {len(channel):>10,} rows")
print(f"  fact_subscriptions : {len(subs):>10,} rows")
print(f"  fact_payments      : {len(pay):>10,} rows")
print(f"  fact_activity_logs : {len(act):>10,} rows")

📊 Files Loaded Successfully!
---------------------------------------------
  dim_users          :     12,000 rows
  dim_plans          :          4 rows
  dim_geography      :         10 rows
  dim_channels       :          6 rows
  fact_subscriptions :     12,000 rows
  fact_payments      :    170,138 rows
  fact_activity_logs :  1,745,584 rows


## 🔍 Cell 4: Data Quality Check Function
Before cleaning, we need to understand what problems exist.
This function checks:
- How many **missing values** (empty cells) each column has
- How many **duplicate rows** exist

In [4]:
def quality_report(df, name):
    """
    Prints a simple quality report for any dataframe.
    Shows missing values and duplicate counts.
    """
    print(f"\n{'='*50}")
    print(f"  Quality Report: {name}")
    print(f"{'='*50}")
    print(f"  Total Rows : {len(df):,}")
    print(f"  Duplicates : {df.duplicated().sum():,}")

    # Check for missing values
    missing = df.isnull().sum()
    missing = missing[missing > 0]   # only show columns WITH missing values

    if len(missing) == 0:
        print("  Missing    : None ✅")
    else:
        print("  Missing values:")
        for col, cnt in missing.items():
            pct = cnt / len(df) * 100
            print(f"    → {col}: {cnt:,} rows ({pct:.1f}% missing)")


# Run the report for each table
quality_report(users, "dim_users")
quality_report(subs,  "fact_subscriptions")
quality_report(pay,   "fact_payments")
quality_report(act,   "fact_activity_logs")


  Quality Report: dim_users
  Total Rows : 12,000
  Duplicates : 0
  Missing values:
    → referral_source: 1,708 rows (14.2% missing)

  Quality Report: fact_subscriptions
  Total Rows : 12,000
  Duplicates : 0
  Missing values:
    → cancel_date: 9,145 rows (76.2% missing)

  Quality Report: fact_payments
  Total Rows : 170,138
  Duplicates : 0
  Missing    : None ✅

  Quality Report: fact_activity_logs
  Total Rows : 1,745,584
  Duplicates : 0
  Missing values:
    → session_duration_min: 1,571,289 rows (90.0% missing)


## 🧹 Cell 5: Clean dim_users
Problems to fix:
1. `signup_date` is stored as text → convert to proper date format
2. String columns may have extra spaces → strip them
3. Duplicate user IDs → remove
4. Missing NPS scores → fill with median

In [5]:
print("🧹 Cleaning: dim_users")
print("-" * 40)

u = users.copy()   # Always work on a COPY, not the original!

# Fix 1: Convert signup_date from text to a proper date
# errors='coerce' means: if a date can't be read, just make it blank (NaT)
u["signup_date"] = pd.to_datetime(u["signup_date"], errors="coerce")
print("  ✅ signup_date converted to datetime")

# Fix 2: Remove extra spaces from all text columns
# Example: '  Male  ' becomes 'Male'
text_columns = u.select_dtypes("object").columns
u[text_columns] = u[text_columns].apply(lambda col: col.str.strip())
print("  ✅ Extra spaces removed from text columns")

# Fix 3: Remove duplicate users (same user_id appearing twice)
rows_before = len(u)
u = u.drop_duplicates(subset="user_id")
rows_removed = rows_before - len(u)
print(f"  ✅ Removed {rows_removed} duplicate rows")

# Fix 4: Fill missing NPS scores with the median value
# Median = the middle value when all scores are sorted
if u["nps_score"].isnull().any():
    median_nps = u["nps_score"].median()
    u["nps_score"] = u["nps_score"].fillna(median_nps)
    print(f"  ✅ Filled missing nps_score with median = {median_nps}")
else:
    print("  ✅ No missing nps_score values")

print(f"\n  📊 Final dim_users: {len(u):,} rows")

# Save cleaned file
u.to_csv(f"{OUTPUT_DIR}/clean_dim_users.csv", index=False)
print(f"  💾 Saved to: {OUTPUT_DIR}/clean_dim_users.csv")

# Preview the cleaned table
u.head(3)

🧹 Cleaning: dim_users
----------------------------------------
  ✅ signup_date converted to datetime
  ✅ Extra spaces removed from text columns
  ✅ Removed 0 duplicate rows
  ✅ No missing nps_score values

  📊 Final dim_users: 12,000 rows
  💾 Saved to: ./data/cleaned/clean_dim_users.csv


,user_id,first_name,last_name,email,age,gender,country,country_id,city,acquisition_channel,channel_id,industry,company_size,signup_date,plan_name,plan_id,is_annual_billing,nps_score,referral_source
0,USR-00001,Arjun,Brown,arjunbrown.38@yahoo.com,39,Male,UK,GEO-002,City_44,Direct,CH-06,Retail,51-200,2021-02-21,Professional,PLN-003,False,8,Friend
1,USR-00002,Charles,Wilson,charleswilson.37@yahoo.com,23,Male,USA,GEO-001,City_36,Organic,CH-01,Healthcare,201-1000,2021-03-07,Basic,PLN-001,True,10,Twitter
2,USR-00003,Karen,Wilson,karenwilson.10@yahoo.com,50,Prefer not to say,India,GEO-004,City_22,Referral,CH-04,Education,1000+,2022-03-28,Basic,PLN-001,True,3,NaN


## 🧹 Cell 6: Clean fact_subscriptions
Problems to fix:
1. Date columns stored as text → convert
2. Rows with no `start_date` → drop (can't use without it)
3. Negative MRR values → set to 0 (revenue can't be negative)

In [6]:
print("🧹 Cleaning: fact_subscriptions")
print("-" * 40)

s = subs.copy()

# Fix 1: Convert date columns from text to proper dates
s["start_date"]  = pd.to_datetime(s["start_date"],  errors="coerce")
s["cancel_date"] = pd.to_datetime(s["cancel_date"], errors="coerce")
print("  ✅ Date columns converted")

# Fix 2: Drop rows where start_date is missing
# We NEED a start date to know when the customer began
rows_before = len(s)
s = s.dropna(subset=["start_date"])
print(f"  ✅ Dropped {rows_before - len(s)} rows with missing start_date")

# Fix 3: Clip negative MRR to 0
# clip(lower=0) means: if value < 0, set it to 0
neg_count = (s["mrr_usd"] < 0).sum()
s["mrr_usd"] = s["mrr_usd"].clip(lower=0)
print(f"  ✅ Fixed {neg_count} negative MRR values (set to 0)")

# Verify: churned column should only contain 0 or 1
assert s["churned"].isin([0, 1]).all(), "ERROR: churned column has unexpected values!"
print(f"  ✅ Churn column verified (0 = stayed, 1 = left)")
print(f"  📊 Overall churn rate: {s['churned'].mean()*100:.1f}%")
print(f"  📊 Final rows: {len(s):,}")

# Save
s.to_csv(f"{OUTPUT_DIR}/clean_fact_subscriptions.csv", index=False)
print(f"  💾 Saved to: {OUTPUT_DIR}/clean_fact_subscriptions.csv")

s.head(3)

🧹 Cleaning: fact_subscriptions
----------------------------------------
  ✅ Date columns converted
  ✅ Dropped 0 rows with missing start_date
  ✅ Fixed 0 negative MRR values (set to 0)
  ✅ Churn column verified (0 = stayed, 1 = left)
  📊 Overall churn rate: 23.8%
  📊 Final rows: 12,000
  💾 Saved to: ./data/cleaned/clean_fact_subscriptions.csv


,subscription_id,user_id,plan_id,plan_name,billing_cycle,monthly_price_usd,mrr_usd,status,start_date,cancel_date,tenure_days,churned,churn_within_30d,churn_within_90d,upgrade_count,downgrade_count,total_revenue_usd,payment_method,auto_renew
0,SUB-000001,USR-00001,PLN-003,Professional,Monthly,79.99,79.99,Active,2021-02-21,NaT,1409,0,0,0,0,0,3756.86,Crypto,True
1,SUB-000002,USR-00002,PLN-001,Basic,Annual,9.99,8.33,Active,2021-03-07,NaT,1395,0,0,0,0,0,387.35,Credit Card,True
2,SUB-000003,USR-00003,PLN-001,Basic,Annual,9.99,8.33,Churned,2022-03-28,2022-10-22,208,1,0,0,0,0,57.75,Crypto,True


## 🧹 Cell 7: Clean fact_payments
Problems to fix:
1. `payment_date` stored as text → convert
2. Extreme/outlier payment amounts → cap using IQR method
3. Duplicate payment IDs → remove

### What is IQR (Interquartile Range)?
> Imagine all amounts sorted lowest to highest. Q1 = value at 25% mark. Q3 = value at 75% mark.
> IQR = Q3 - Q1. Anything **3× outside** this range is an outlier — we cap it instead of deleting.

In [7]:
print("🧹 Cleaning: fact_payments")
print("-" * 40)

p = pay.copy()

# Fix 1: Convert payment_date to datetime
p["payment_date"] = pd.to_datetime(p["payment_date"], errors="coerce")
print("  ✅ payment_date converted to datetime")

# Fix 2: Cap outliers in amount_usd using the IQR method
Q1  = p["amount_usd"].quantile(0.25)   # Lower quarter
Q3  = p["amount_usd"].quantile(0.75)   # Upper quarter
IQR = Q3 - Q1                           # The middle spread

lower_cap = Q1 - 3 * IQR   # Anything below this is too low
upper_cap = Q3 + 3 * IQR   # Anything above this is too high

outlier_count = ((p["amount_usd"] < lower_cap) | (p["amount_usd"] > upper_cap)).sum()
p["amount_usd"] = p["amount_usd"].clip(lower=lower_cap, upper=upper_cap)
print(f"  ✅ Capped {outlier_count} outlier payment amounts")
print(f"     (Range kept: ${lower_cap:.2f}  to  ${upper_cap:.2f})")

# Fix 3: Remove duplicate payment rows
rows_before = len(p)
p = p.drop_duplicates(subset="payment_id")
print(f"  ✅ Removed {rows_before - len(p)} duplicate payment rows")

print(f"  📊 Final rows: {len(p):,}")

# Save
p.to_csv(f"{OUTPUT_DIR}/clean_fact_payments.csv", index=False)
print(f"  💾 Saved to: {OUTPUT_DIR}/clean_fact_payments.csv")

p.head(3)

🧹 Cleaning: fact_payments
----------------------------------------
  ✅ payment_date converted to datetime
  ✅ Capped 6430 outlier payment amounts
     (Range kept: $-200.01  to  $289.99)
  ✅ Removed 0 duplicate payment rows
  📊 Final rows: 170,138
  💾 Saved to: ./data/cleaned/clean_fact_payments.csv


,payment_id,user_id,subscription_id,payment_date,amount_usd,billing_cycle,payment_method,payment_status,is_failed,is_refunded,invoice_id,currency,discount_applied,discount_pct
0,PAY-0000001,USR-00001,SUB-000001,2021-02-21,79.99,Monthly,Crypto,Success,0,0,INV-0000001,USD,0,5
1,PAY-0000002,USR-00001,SUB-000001,2021-03-23,79.99,Monthly,Crypto,Success,0,0,INV-0000002,USD,0,0
2,PAY-0000003,USR-00001,SUB-000001,2021-04-22,79.99,Monthly,Crypto,Success,0,0,INV-0000003,USD,0,0


## 🧹 Cell 8: Clean fact_activity_logs
Problems to fix:
1. `event_date` stored as text → convert
2. Missing `session_duration_min` → fill with 0 (no session recorded)
3. Duplicate activity IDs → remove

In [8]:
print("🧹 Cleaning: fact_activity_logs")
print("-" * 40)

a = act.copy()

# Fix 1: Convert event_date to datetime
a["event_date"] = pd.to_datetime(a["event_date"], errors="coerce")
print("  ✅ event_date converted to datetime")

# Fix 2: Fill missing session duration with 0
# If no session was recorded, we treat it as 0 minutes
missing_sessions = a["session_duration_min"].isnull().sum()
a["session_duration_min"] = a["session_duration_min"].fillna(0)
print(f"  ✅ Filled {missing_sessions:,} missing session durations with 0")

# Fix 3: Remove duplicate activity rows
rows_before = len(a)
a = a.drop_duplicates(subset="activity_id")
print(f"  ✅ Removed {rows_before - len(a)} duplicate activity rows")

print(f"  📊 Final rows: {len(a):,}")

# Save
a.to_csv(f"{OUTPUT_DIR}/clean_fact_activity_logs.csv", index=False)
print(f"  💾 Saved to: {OUTPUT_DIR}/clean_fact_activity_logs.csv")

a.head(3)

🧹 Cleaning: fact_activity_logs
----------------------------------------
  ✅ event_date converted to datetime
  ✅ Filled 1,571,289 missing session durations with 0
  ✅ Removed 0 duplicate activity rows
  📊 Final rows: 1,745,584
  💾 Saved to: ./data/cleaned/clean_fact_activity_logs.csv


,activity_id,user_id,subscription_id,event_date,event_type,feature_used,session_duration_min,device_type,os,is_mobile,page_views,error_occurred
0,ACT-00000001,USR-00001,SUB-000001,2023-07-22,api_call,reports,0.0,Mobile,Windows,1,2,0
1,ACT-00000002,USR-00001,SUB-000001,2021-03-10,dashboard_view,mobile_app,0.0,Mobile,Windows,1,13,0
2,ACT-00000003,USR-00001,SUB-000001,2022-04-12,export,team_collab,0.0,Mobile,Windows,1,6,0


## 💾 Cell 9: Save Dimension Tables
The dimension tables (plans, geography, channels) don't need major cleaning — just save them to the cleaned folder.

In [9]:
# Save dimension tables as-is (they are already clean)
plans.to_csv(f"{OUTPUT_DIR}/clean_dim_plans.csv",   index=False)
geo.to_csv(f"{OUTPUT_DIR}/clean_dim_geography.csv", index=False)
channel.to_csv(f"{OUTPUT_DIR}/clean_dim_acquisition_channel.csv", index=False)

print("💾 Dimension tables saved:")
print(f"  ✅ clean_dim_plans.csv")
print(f"  ✅ clean_dim_geography.csv")
print(f"  ✅ clean_dim_acquisition_channel.csv")

💾 Dimension tables saved:
  ✅ clean_dim_plans.csv
  ✅ clean_dim_geography.csv
  ✅ clean_dim_acquisition_channel.csv


## ✅ Cell 10: Final Summary
Let's verify all 7 cleaned files were saved correctly.

In [10]:
# List all files saved in the cleaned folder
saved_files = os.listdir(OUTPUT_DIR)

print("="*50)
print("✅ STEP 1 COMPLETE — All cleaned files saved!")
print("="*50)
print(f"\n📂 Files in {OUTPUT_DIR}:")
for f in sorted(saved_files):
    print(f"   ✓ {f}")

print("\n👉 Next Step: Run step02_build_mat.ipynb")

✅ STEP 1 COMPLETE — All cleaned files saved!

📂 Files in ./data/cleaned:
   ✓ clean_dim_acquisition_channel.csv
   ✓ clean_dim_geography.csv
   ✓ clean_dim_plans.csv
   ✓ clean_dim_users.csv
   ✓ clean_fact_activity_logs.csv
   ✓ clean_fact_payments.csv
   ✓ clean_fact_subscriptions.csv

👉 Next Step: Run step02_build_mat.ipynb
